# Breast Cancer Diagnosis Prediction — ML Classifier

**Problem:** Misdiagnosis or delayed diagnosis of breast cancer costs lives. This notebook builds and compares
machine learning models that classify tumors as **malignant** or **benign** from digitized fine needle
aspirate (FNA) measurements, and packages the best model for deployment.

**Dataset:** Breast Cancer Wisconsin (Diagnostic), 569 samples, 30 numeric features — a well-known
public benchmark (available directly via `scikit-learn`, no external download needed).

**Pipeline:** EDA → train/test split → scaling → train 3 model families → compare with
cross-validation → pick the best model by F1 → explain it with feature importance → export `model.pkl`
for the Streamlit app in this repo.

> Run all cells top to bottom in Google Colab. No dataset upload required.

In [ ]:
!pip -q install xgboost shap joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, RocCurveDisplay, classification_report
)
from xgboost import XGBClassifier
import joblib

sns.set_theme(style="whitegrid")

## 1. Load & explore the data

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")  # 0 = malignant, 1 = benign

print(X.shape)
print(y.value_counts().rename({0: "malignant", 1: "benign"}))
X.head()

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x=y.map({0: "malignant", 1: "benign"}))
plt.title("Class balance")
plt.xlabel("")
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(X.iloc[:, :10].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation — first 10 features")
plt.show()

## 2. Train/test split & scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Train & compare three model families

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        eval_metric="logloss", random_state=42
    ),
}

results = {}
fitted = {}

for name, model in models.items():
    uses_scaled = name == "Logistic Regression"
    Xtr, Xte = (X_train_s, X_test_s) if uses_scaled else (X_train, X_test)

    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    probs = model.predict_proba(Xte)[:, 1]
    cv = cross_val_score(model, Xtr, y_train, cv=5, scoring="f1").mean()

    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, probs),
        "cv_f1_mean": cv,
    }
    fitted[name] = model

results_df = pd.DataFrame(results).T.round(4)
results_df

In [ ]:
results_df[["f1", "roc_auc", "cv_f1_mean"]].plot(kind="bar", figsize=(8, 5))
plt.title("Model comparison")
plt.ylim(0.85, 1.0)
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

## 4. Pick the best model & inspect errors

In [ ]:
best_name = results_df["f1"].idxmax()
best_model = fitted[best_name]
uses_scaled = best_name == "Logistic Regression"
Xte_best = X_test_s if uses_scaled else X_test

print("Best model:", best_name)
print(classification_report(y_test, best_model.predict(Xte_best), target_names=["malignant", "benign"]))

cm = confusion_matrix(y_test, best_model.predict(Xte_best))
plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["malignant", "benign"], yticklabels=["malignant", "benign"])
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.title(f"Confusion matrix — {best_name}")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(best_model, Xte_best, y_test)
plt.title(f"ROC curve — {best_name}")
plt.show()

## 5. Explainability — which features drive the prediction?

In [ ]:
if best_name in ("Random Forest", "XGBoost"):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
    importances.sort_values(ascending=False).head(10).plot(kind="barh", figsize=(7, 5))
    plt.title(f"Top 10 feature importances — {best_name}")
    plt.gca().invert_yaxis()
    plt.show()
else:
    coefs = pd.Series(best_model.coef_[0], index=X.columns)
    coefs.abs().sort_values(ascending=False).head(10).index.to_series().apply(
        lambda f: coefs[f]
    ).sort_values().plot(kind="barh", figsize=(7, 5))
    plt.title(f"Top 10 |coefficient| features — {best_name}")
    plt.show()

## 6. Export the model for deployment

Saves everything the Streamlit app (`app.py` in this repo) needs: the fitted model, the scaler
(only used for Logistic Regression), and the feature name order.

In [ ]:
joblib.dump(
    {
        "model": best_model,
        "scaler": scaler if uses_scaled else None,
        "feature_names": list(X.columns),
        "model_name": best_name,
        "metrics": results[best_name],
    },
    "model.pkl",
)
print("Saved model.pkl — download it and commit it alongside app.py in your GitHub repo.")

## Next steps for the GitHub repo

1. Download `model.pkl` from the Colab file browser (left sidebar → folder icon).
2. Place it in the project folder next to `app.py`, `requirements.txt`, and `README.md`.
3. Push to GitHub and (optionally) deploy the Streamlit app for free on Streamlit Community Cloud —
   see `README.md` for the exact commands.